# 01. Xe II: A, силы осцилляторов и ветвления для верхних уровней выбранных линий

Линии и номера уровней (Wang et al 2019, лист «NIST Level Table») — `nb.XE2_LINES`
(обоснование выбора: `docs/crm/lines_final.md`).

У этих уровней в NIST известны A лишь для 0–3 ветвей из 14–27 разрешённых, поэтому все
ветви считаются в DBSR: мишень как у Ванга (`models.xe2_wang`), затем `dbsr_mult3` +
`dbsr_dmat3` для каждой пары «верхний уровень — нижний уровень» (E1). A пересчитываются
на экспериментальные энергии (NIST, через таблицу уровней Ванга): A ∝ S·ΔE³.

Ветви считаются для всех 19 уровней 5p⁴6p (секунды), чтобы сравнить с каждым A из NIST
(у выбранных уровней их всего 5). Расчёт только структурный (без рассеяния), идёт отдельным
процессом; замер на 4 ядрах: мишень ~5 мин, переходы ~10 с.

**Проверки**: энергии; все A против NIST (21 линия); отношение калибровок скорость/длина;
отношения A для линий с общим верхним уровнем против измеренных отношений интенсивностей
(спектр A3, `docs/crm/criteria_A3.csv`). Ветвления выдаются в двух вариантах: только DBSR
и «A из NIST там, где они есть, остальные ветви из DBSR».

**Внимание**: в тестовом прогоне ветви уровня 44 (линия 545.045) противоречили спектру,
поэтому 545.045 заменена на 557.219 (уровень 42). Для уровней 29, 39, 41, 42, 51 расчёт
со спектром и NIST согласуется (`docs/crm/lines_final.md`).

**Выход**: `xe2_wang/transitions_E1.csv` (все рассчитанные ветви) и
`xe2_wang/xe2_A_branching.xlsx` / `.csv` (ветви выбранных верхних уровней: λ, A, gf, S,
отношение калибровок скорость/длина, ветвление, A NIST там, где оно есть).

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
ROOT    = Path("/data/files_Otdel120/pydbsr_calc")    # папка установки (большой диск)
REPO    = ROOT / "pydbsr_src"                         # распакованный архив pydbsr (или клон репозитория)
CRM     = REPO / "examples" / "crm"                   # nbtools.py, run_model.py, models.py
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST ASD (nist_*.csv)
DATA    = ROOT                                        # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
RUNS    = ROOT / "runs" / "crm"                       # результаты расчётов (большой диск)
XE2_RUN = ROOT / "runs" / "xe_plus_v2"                # готовый расчёт e + Xe+ (модель Ванга, J <= 25)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
TE = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0]   # сетка Te (эВ) для констант скоростей
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import rates, reference
import nbtools as nb
%matplotlib inline

In [ ]:
UPPERS = sorted(set(nb.XE2_UPPERS) | set(nb.XE2_VALIDATION_UPPERS))   # выбранные + все 6p для проверки
job = nb.Job(RUNS / "xe2_wang", "xe2_wang", levels=XLSX, stage="transitions",
             uppers=",".join(map(str, UPPERS)), cores=CORES, multipoles="E1")
job.start()

In [ ]:
job.status()          # ход расчёта (запускайте когда хотите)

## Энергии мишени против NIST (проверка мишени)

In [ ]:
W = RUNS / "xe2_wang"
print((W / "target_table.txt").read_text())

## Ветви уровней 6p: DBSR и NIST

`A_exp` — A при экспериментальной энергии перехода; `vel/len` — отношение A в калибровках
скорости и длины (далеко от 1 — ненадёжная ветвь); `A_NIST` — из `docs/crm/nist_XeII*.csv`.

In [ ]:
import csv
ref = reference.read_xlsx(XLSX)
elv = [lv.energy_cm for lv in ref.levels]
nistA = {}
for f in ("nist_XeII.csv", "nist_XeII_UV.csv", "nist_XeII_IR.csv"):
    if (NIST / f).exists():
        for x in nb.nist_lines(NIST / f):
            u, l = nb.level_number(elv, x["Ek"]), nb.level_number(elv, x["Ei"])
            if u and l and x["A"]:
                nistA[(u, l)] = x["A"]
rows = []
for r in nb.read_csv(W / "transitions_E1.csv"):
    if not r["upper_no"] or not r["lower_no"]:
        continue
    u, l = int(r["upper_no"]), int(r["lower_no"])
    rows.append(dict(upper=u, upper_label=ref.label(u), lower=l, lower_label=ref.label(l),
                     wl_air_nm=nb.fnum(r["wl_air_nm"]), A_exp=float(r["A_exp"]), A_calc=float(r["A_calc"]),
                     gf=float(r["gf"]), S=float(r["S"]), vel_len=nb.fnum(r["gauge_ratio"]),
                     A_NIST=nistA.get((u, l))))
tot, tot_h = {}, {}
for r in rows:
    r["A_hybrid"] = r["A_NIST"] or r["A_exp"]            # NIST там, где есть, иначе DBSR
    tot[r["upper"]] = tot.get(r["upper"], 0) + r["A_exp"]
    tot_h[r["upper"]] = tot_h.get(r["upper"], 0) + r["A_hybrid"]
for r in rows:
    r["branching"] = r["A_exp"] / tot[r["upper"]]               # только DBSR
    r["branching_hybrid"] = r["A_hybrid"] / tot_h[r["upper"]]   # A NIST, где известны
    r["tau_ns"] = 1e9 / tot[r["upper"]]
rows.sort(key=lambda r: (r["upper"], -r["A_exp"]))
cmp = np.array([(r["A_exp"], r["A_NIST"]) for r in rows if r["A_NIST"]])
q = cmp[:, 0] / cmp[:, 1]
print(f"все A с NIST: {len(q)} линий, медиана DBSR/NIST = {np.median(q):.2f}, в пределах ×2: {np.sum((q > 0.5) & (q < 2))}")
plt.loglog(cmp[:, 1], cmp[:, 0], "o"); x = np.array([1e6, 3e8]); plt.plot(x, x, "k-", x, 2 * x, "k:", x, x / 2, "k:")
plt.xlabel("A NIST, с⁻¹"); plt.ylabel("A DBSR, с⁻¹"); plt.show()
for u in nb.XE2_UPPERS:
    print(f"\n{u} {ref.label(u)}   ΣA = {tot[u]:.3e} с-1  (τ = {1e9 / tot[u]:.2f} нс)")
    for r in [r for r in rows if r["upper"] == u]:
        nist = f"NIST {r['A_NIST']:.2e} ({r['A_exp'] / r['A_NIST']:.2f})" if r["A_NIST"] else ""
        print(f"   -> {r['lower']:3d} {r['lower_label']:26s} {r['wl_air_nm'] or 0:9.3f} нм  A = {r['A_exp']:.3e}  "
              f"BR = {r['branching']:.4f}  gf = {r['gf']:.3e}  vel/len = {r['vel_len']}  {nist}")

## Проверка по спектру: линии с общим верхним уровнем

Отношение числа фотонов двух линий одного верхнего уровня равно отношению их A (CRM не
нужна). Измеренные амплитуды — спектр A3 (`docs/crm/criteria_A3.csv`, лист `SHEET_CHECK`,
выдержка 0.4 с: насыщенных линий нет); для 545.045/545.090 — фит дублета
(`criteria_A3_doublet.csv`). Калибровки спектральной чувствительности нет, поэтому
сравнивайте прежде всего линии, близкие по длине волны.

In [ ]:
SHEET_CHECK = "data10"
crit = {round(float(r["wl_nm"]), 3): r for r in nb.read_csv(NIST / "criteria_A3.csv") if r["ion"] == "XeII"}
d2 = [d for d in nb.read_csv(NIST / "criteria_A3_doublet.csv") if d["sheet"] == SHEET_CHECK][0]
meas = {545.045: float(d2["amp1"]), 545.090: float(d2["amp2"])}
for wl, r in crit.items():
    if r.get(f"{SHEET_CHECK}_amp") and not r.get(f"{SHEET_CHECK}_sat") and wl not in meas:
        meas[wl] = float(r[f"{SHEET_CHECK}_amp"])
A_of, A_hyb = {}, {}
for wl, x in crit.items():
    if x.get("upper_no") and x.get("lower_no"):
        k = (int(x["upper_no"]), int(x["lower_no"]))
        a = [r for r in rows if (r["upper"], r["lower"]) == k]
        if a:
            A_of[wl], A_hyb[wl] = a[0]["A_exp"], a[0]["A_hybrid"]
print(f"{'пара':>18s}  {'измерено':>9s}  {'DBSR':>8s}  {'NIST/DBSR':>9s}")
for w1, w2 in nb.XE2_SAME_UPPER:
    if w1 in A_of and w2 in A_of:
        m = meas[w2] / meas[w1] if w1 in meas and w2 in meas else float("nan")
        print(f"{w2:8.3f}/{w1:8.3f}  {m:9.2f}  {A_of[w2] / A_of[w1]:8.2f}  {A_hyb[w2] / A_hyb[w1]:9.2f}")

In [ ]:
import openpyxl
out = W / "xe2_A_branching.csv"
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
wb = openpyxl.Workbook(); ws = wb.active; ws.title = "branches"
ws.append(list(rows[0]))
for r in rows:
    ws.append(list(r.values()))
ws2 = wb.create_sheet("selected lines")
ws2.append(["wl_nm (NIST)", "upper", "lower", "role", "A_exp (DBSR)", "branching (DBSR)", "A_NIST",
            "branching (NIST A где есть, иначе DBSR)"])
for x in nb.XE2_LINES:
    r = [r for r in rows if r["upper"] == x["upper"] and r["lower"] == x["lower"]]
    ws2.append([x["wl"], x["upper"], x["lower"], x["role"], r[0]["A_exp"] if r else None,
                r[0]["branching"] if r else None, nistA.get((x["upper"], x["lower"])),
                r[0]["branching_hybrid"] if r else None])
wb.save(W / "xe2_A_branching.xlsx")
print("->", out, "и", W / "xe2_A_branching.xlsx")